# 02: Execute PubMed Searches & Identify Excluded Papers

## Objective
Execute the translated PubMed search queries (from notebook 00) against the PubMed API
to identify papers that were retrieved by systematic review searches but are **NOT** referenced
in any Cochrane review. These form the **excluded** set for ground truth evaluation.

## Revised Ground Truth Logic
- **Included (label=1)**: ALL papers referenced in any Cochrane review
  (regardless of internal categorization as included/excluded/awaiting/ongoing)
- **Excluded (label=0)**: Papers that appear in PubMed search results but are
  NOT referenced in any Cochrane review

## Pipeline
1. Load translated search strategies from `search_strategies.csv`
2. Build "known papers" set from Cochrane reviews (PMIDs from notebooks 00 + 01)
3. Execute each PubMed query via NCBI Entrez API
4. Collect PMIDs from search results, track source review
5. Filter out papers that appear in any Cochrane review
6. Fetch abstracts for excluded papers
7. Save results

## Input Files
- `Data/search_strategies.csv` — Translated PubMed queries (from notebook 00)
- `Data/categorized_references.csv` — References extracted from PDFs (notebook 00)
- `Data/referenced_paper_abstracts.csv` — Matched references with PMIDs (notebook 01)
- `Data/doi_pmid_cache.csv` — DOI→PMID mappings (notebook 01)

## Output Files
- `Data/pubmed_search_results.csv` — Search execution log (counts, errors, PMIDs per query)
- `Data/pubmed_excluded_abstracts.csv` — Abstracts for excluded-only papers

In [1]:
%pip install -q biopython pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import time
import json
import re
from pathlib import Path
from tqdm.notebook import tqdm
from Bio import Entrez
from collections import Counter, defaultdict

# =============================================================================
# Configuration
# =============================================================================

# NCBI API — with API key allows 10 requests/sec
Entrez.email = os.environ.get("NCBI_EMAIL", "")
Entrez.api_key = os.environ.get("NCBI_API_KEY", "")

NCBI_RATE = 0.11 if Entrez.api_key else 0.34  # seconds between requests
SEARCH_RETMAX = 100000  # max PMIDs to retrieve per query

# Paths
notebook_dir = Path.cwd()
project_root = notebook_dir if (notebook_dir / "Data").exists() else notebook_dir.parent
DATA_DIR = project_root / "Data"

# Input files
STRATEGIES_CSV  = DATA_DIR / "search_strategies.csv"
REFS_CSV        = DATA_DIR / "categorized_references.csv"
ABSTRACTS_CSV   = DATA_DIR / "referenced_paper_abstracts.csv"
DOI_PMID_CACHE  = DATA_DIR / "doi_pmid_cache.csv"
METADATA_CSV    = DATA_DIR / "review_metadata.csv"

# Output files
SEARCH_RESULTS_CSV    = DATA_DIR / "pubmed_search_results.csv"
SEARCH_PROGRESS_CSV   = DATA_DIR / "pubmed_search_progress.csv"
EXCLUDED_ABSTRACTS_CSV = DATA_DIR / "pubmed_excluded_abstracts.csv"

print(f"Data directory: {DATA_DIR}")
print(f"NCBI API key configured: {bool(Entrez.api_key)}")
print(f"Rate limit: {1/NCBI_RATE:.0f} requests/sec")
print(f"Max PMIDs per query: {SEARCH_RETMAX:,}")

Data directory: c:\Users\juanx\Documents\LSE-UKHSA Project\Data
NCBI API key configured: True
Rate limit: 9 requests/sec
Max PMIDs per query: 100,000


In [3]:
# =============================================================================
# Load Search Strategies
# =============================================================================

strategies = pd.read_csv(STRATEGIES_CSV)
print(f"Total strategies: {len(strategies):,}")
print(f"\nTranslation notes:")
print(strategies['translation_notes'].value_counts().head(10).to_string())

# ── Version deduplication (keep latest version of each Cochrane review) ───────
META_CSV = DATA_DIR / "review_metadata.csv"
meta = pd.read_csv(META_CSV)

if 'cd_number' not in meta.columns or 'version' not in meta.columns:
    _vp = meta['doi'].str.extract(r'(CD\d+)(?:\.pub(\d+))?', flags=re.I)
    meta['cd_number'] = _vp[0].str.upper()
    meta['version'] = _vp[1].fillna(1).astype(int)

_has_cd = meta[meta['cd_number'].notna()]
_latest_idx = _has_cd.groupby('cd_number')['version'].idxmax()
_no_cd = meta[meta['cd_number'].isna()]
meta = pd.concat([meta.loc[_latest_idx], _no_cd], ignore_index=True)

latest_dois = set(meta['doi'].dropna())
strategies = strategies[strategies['doi'].isin(latest_dois)].copy()
print(f"\nAfter version dedup: {len(strategies):,} strategies ({len(latest_dois):,} reviews)")

# Filter to successfully translated queries
translated = strategies[strategies['pubmed_query'].notna()].copy()
print(f"Translated queries: {len(translated):,}")

# Deduplicate identical queries (different versions of the same review often share a search)
translated['query_hash'] = translated['pubmed_query'].apply(hash)
unique_queries = translated.drop_duplicates(subset='query_hash').copy()

# Build mapping: query_hash → list of review DOIs
query_to_reviews = translated.groupby('query_hash')['doi'].apply(list).to_dict()

print(f"Unique queries after dedup: {len(unique_queries):,}")
print(f"  (covers {len(translated):,} review-strategy pairs)")
print(f"\nQuery length stats:")
print(unique_queries['pubmed_query'].str.len().describe().to_string())

Total strategies: 126

Translation notes:
translation_notes
narrative_only                                       67
filtered_not_a_query                                 15
adjacency_converted_to_AND                           14
success                                              13
mp_approximated                                       4
adjacency_converted_to_AND; wildcard_approximated     3
wildcard_approximated; near_converted_to_AND          1
adjacency_converted_to_AND; mp_approximated           1
near_converted_to_AND                                 1
wildcard_approximated                                 1

After version dedup: 126 strategies (119 reviews)
Translated queries: 42
Unique queries after dedup: 40
  (covers 42 review-strategy pairs)

Query length stats:
count      40.000000
mean      278.475000
std       377.458165
min        11.000000
25%        65.500000
50%       133.500000
75%       296.250000
max      1514.000000


In [4]:
# =============================================================================
# Build Known PMID Set (papers already in Cochrane reviews)
# =============================================================================
# Union of all PMIDs from: direct extraction (notebook 01), CrossRef matching
# (notebook 02), and DOI→PMID cache (notebook 02).

known_pmids = set()

# Source 1: Direct PMIDs from reference extraction
if REFS_CSV.exists():
    refs = pd.read_csv(REFS_CSV, usecols=['pmid'], dtype=str)
    direct = {p.strip() for p in refs['pmid'].dropna() if p.strip().isdigit()}
    known_pmids |= direct
    print(f"Source 1 — Direct extraction:    {len(direct):>8,} PMIDs")

# Source 2: Matched PMIDs from abstract fetching
if ABSTRACTS_CSV.exists():
    abs_df = pd.read_csv(ABSTRACTS_CSV, usecols=['pmid'], dtype=str)
    matched = {p.strip() for p in abs_df['pmid'].dropna() if p.strip().isdigit()}
    known_pmids |= matched
    print(f"Source 2 — Matched abstracts:    {len(matched):>8,} PMIDs")

# Source 3: DOI→PMID cache
if DOI_PMID_CACHE.exists():
    cache = pd.read_csv(DOI_PMID_CACHE, dtype=str)
    cached = {p.strip() for p in cache[cache['pmid'] != 'NO_PMID']['pmid'].dropna() if p.strip().isdigit()}
    known_pmids |= cached
    print(f"Source 3 — DOI→PMID cache:      {len(cached):>8,} PMIDs")

print(f"\n{'='*50}")
print(f"Total known PMIDs (in Cochrane reviews): {len(known_pmids):,}")

# ── Load Review Publication Years (for date filtering) ──────────────────────
# Years are extracted from the PDF citation line (NB00) and stored in
# review_metadata.csv with CD numbers as keys — matching review_doi format.
review_meta = pd.read_csv(METADATA_CSV, usecols=['doi', 'year'], dtype=str)
review_meta['year'] = pd.to_numeric(review_meta['year'], errors='coerce')
review_meta = review_meta.dropna(subset=['year'])
review_year_map = dict(zip(review_meta['doi'], review_meta['year'].astype(int)))
print(f"\nReview publication years loaded: {len(review_year_map):,} reviews")
print(f"Year range: {min(review_year_map.values())} - {max(review_year_map.values())}")

Source 1 — Direct extraction:         120 PMIDs
Source 2 — Matched abstracts:       9,177 PMIDs
Source 3 — DOI→PMID cache:         8,215 PMIDs

Total known PMIDs (in Cochrane reviews): 9,244

Review publication years loaded: 119 reviews
Year range: 1998 - 2025


In [5]:
# =============================================================================
# PubMed Search & Abstract Fetch Functions
# =============================================================================

def execute_pubmed_search(query, retmax=10000, max_retries=3, sort="relevance"):
    """Execute a PubMed search and return PMIDs + total count.

    Args:
        sort: "relevance" for most relevant first, "pub_date" for newest first

    Returns:
        dict with keys: pmids (list[str]), count (int), error (str|None),
              retmax_hit (bool — True if count > retmax, results truncated)
    """
    for attempt in range(max_retries):
        try:
            handle = Entrez.esearch(
                db="pubmed",
                term=query,
                retmax=retmax,
                usehistory="y",
                sort=sort,  # "relevance" = most relevant first
            )
            results = Entrez.read(handle)
            handle.close()

            count = int(results.get('Count', 0))
            pmids = [str(p) for p in results.get('IdList', [])]

            return {
                'pmids': pmids,
                'count': count,
                'error': None,
                'retmax_hit': count > retmax,
            }
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** (attempt + 1))
            else:
                return {'pmids': [], 'count': 0, 'error': str(e), 'retmax_hit': False}


def extract_pubmed_record(record):
    """Extract key fields from a PubMed XML article record."""
    try:
        cit = record['MedlineCitation']
        art = cit['Article']
        pmid = str(cit['PMID'])
        title = str(art.get('ArticleTitle', ''))

        # Abstract
        abstract = ''
        if 'Abstract' in art and 'AbstractText' in art['Abstract']:
            parts = art['Abstract']['AbstractText']
            abstract = ' '.join(str(p) for p in parts) if isinstance(parts, list) else str(parts)

        # Year
        year = ''
        if 'Journal' in art and 'JournalIssue' in art['Journal']:
            year = art['Journal']['JournalIssue'].get('PubDate', {}).get('Year', '')

        # Authors
        authors = []
        if 'AuthorList' in art:
            for auth in art['AuthorList']:
                if 'LastName' in auth:
                    name = auth['LastName']
                    if auth.get('Initials'):
                        name += ' ' + auth['Initials']
                    authors.append(name)

        # DOI
        doi = ''
        if 'ELocationID' in art:
            for loc in art['ELocationID']:
                if hasattr(loc, 'attributes') and loc.attributes.get('EIdType') == 'doi':
                    doi = str(loc)
                    break

        return {'pmid': pmid, 'title': title, 'abstract': abstract,
                'year': year, 'authors': '; '.join(authors), 'doi': doi}
    except Exception:
        return None


def fetch_abstracts_batch(pmids, batch_size=200, max_retries=3):
    """Batch-fetch PubMed records. Returns {pmid: record_dict}."""
    results = {}
    pmid_list = [str(p) for p in pmids if str(p).isdigit()]

    batches = range(0, len(pmid_list), batch_size)
    for i in tqdm(batches, desc="Fetching abstracts"):
        batch = pmid_list[i:i + batch_size]

        for attempt in range(max_retries):
            try:
                time.sleep(NCBI_RATE * (attempt + 1))
                handle = Entrez.efetch(
                    db="pubmed", id=",".join(batch),
                    rettype="xml", retmode="xml"
                )
                records = Entrez.read(handle)
                handle.close()

                for article in records.get('PubmedArticle', []):
                    data = extract_pubmed_record(article)
                    if data:
                        results[data['pmid']] = data
                break  # success
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"  Batch {i//batch_size + 1} failed after {max_retries} attempts: {e}")

    return results


print("Functions defined:")
print("  • execute_pubmed_search() — Run a PubMed query, get PMIDs")
print("  • fetch_abstracts_batch()  — Batch fetch abstracts by PMID")

Functions defined:
  • execute_pubmed_search() — Run a PubMed query, get PMIDs
  • fetch_abstracts_batch()  — Batch fetch abstracts by PMID


In [6]:
# =============================================================================
# Execute PubMed Searches
# =============================================================================

print("EXECUTING PUBMED SEARCHES")
print("=" * 60)
print(f"Queries to execute: {len(unique_queries):,}")
print(f"Estimated time: ~{len(unique_queries) * NCBI_RATE / 60:.0f} min")

# --- Main loop ---
batch_results = []
total_pmids = 0
errors = 0
capped = 0
start_time = time.time()

for idx, (_, row) in enumerate(tqdm(unique_queries.iterrows(),
                                     total=len(unique_queries), desc="PubMed Search")):
    query = row['pubmed_query']
    review_dois = query_to_reviews.get(row['query_hash'], [row['doi']])

    result = execute_pubmed_search(query, retmax=SEARCH_RETMAX)
    time.sleep(NCBI_RATE)

    batch_results.append({
        'query_hash': row['query_hash'],
        'review_doi': row['doi'],
        'all_review_dois': '|'.join(review_dois),
        'query_length': len(query),
        'result_count': result['count'],
        'pmids_retrieved': len(result['pmids']),
        'retmax_hit': result['retmax_hit'],
        'error': result['error'],
        'pmids_json': json.dumps(result['pmids']),
    })

    total_pmids += len(result['pmids'])
    if result['error']:
        errors += 1
    if result['retmax_hit']:
        capped += 1

    # Print progress every 10 queries
    if (idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed * 60
        print(f"  [{idx+1:,}/{len(unique_queries):,}] "
              f"{total_pmids:,} PMIDs | {errors} errors | "
              f"{capped} capped | {rate:.0f} q/min")

# Save final results
results_df = pd.DataFrame(batch_results)
results_df.to_csv(SEARCH_PROGRESS_CSV, index=False)

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"Complete: {len(unique_queries):,} queries in {elapsed/60:.1f} min")
print(f"  Total PMIDs retrieved: {total_pmids:,}")
print(f"  Errors: {errors:,}")
print(f"  Queries exceeding retmax ({SEARCH_RETMAX:,}): {capped:,}")

EXECUTING PUBMED SEARCHES
Queries to execute: 40
Estimated time: ~0 min


PubMed Search:   0%|          | 0/40 [00:00<?, ?it/s]

  [10/40] 46,544 PMIDs | 0 errors | 2 capped | 27 q/min
  [20/40] 110,576 PMIDs | 0 errors | 4 capped | 27 q/min
  [30/40] 154,462 PMIDs | 0 errors | 6 capped | 26 q/min
  [40/40] 239,765 PMIDs | 0 errors | 11 capped | 23 q/min

Complete: 40 queries in 1.7 min
  Total PMIDs retrieved: 239,765
  Errors: 0
  Queries exceeding retmax (100,000): 11


In [7]:
# =============================================================================
# Filter Queries & Identify Excluded PMIDs
# =============================================================================

print("IDENTIFYING EXCLUDED PAPERS")
print("=" * 60)

# Use search results from previous cell (also saved to SEARCH_PROGRESS_CSV)
print(f"Total queries executed: {len(results_df):,}")
print(f"  With results: {(results_df['result_count'] > 0).sum():,}")
print(f"  Errors: {results_df['error'].notna().sum():,}")
print(f"  Hit retmax cap ({SEARCH_RETMAX:,}): {results_df['retmax_hit'].sum():,}")

# ---- QUALITY FILTER ----
# Keep all queries that returned results (no errors).
# Queries that hit the retmax cap ARE still useful — the PMIDs we
# retrieved are valid search results that the review chose not to include.
# Discarding them would leave us with almost no excluded papers.

successful = results_df[
    (results_df['error'].isna()) &
    (results_df['result_count'] > 0)
].copy()

n_capped = successful['retmax_hit'].sum()
print(f"\nAfter quality filter (non-empty, no errors):")
print(f"  Retained queries: {len(successful):,}")
print(f"  Of which hit retmax cap: {n_capped:,}")
print(f"  Dropped (empty/errored): {len(results_df) - len(successful):,}")
print(f"\nResult count stats for retained queries:")
print(successful['result_count'].describe().to_string())

# Collect all unique PMIDs and track source reviews
all_search_pmids = set()
pmid_to_reviews = defaultdict(set)  # pmid → set of review DOIs

for _, row in tqdm(successful.iterrows(), total=len(successful), desc="Collecting PMIDs"):
    if pd.isna(row['pmids_json']) or row['pmids_json'] == '[]':
        continue
    pmids = json.loads(row['pmids_json'])
    review_dois = row['all_review_dois'].split('|')
    for pmid in pmids:
        all_search_pmids.add(pmid)
        for rdoi in review_dois:
            pmid_to_reviews[pmid].add(rdoi)

print(f"\nTotal unique PMIDs from filtered searches: {len(all_search_pmids):,}")

# Partition into included (in Cochrane) and excluded (search-only)
overlap_pmids  = all_search_pmids & known_pmids
excluded_pmids = all_search_pmids - known_pmids

print(f"\nOverlap with Cochrane reviews:  {len(overlap_pmids):,} PMIDs")
print(f"Excluded (search-only):         {len(excluded_pmids):,} PMIDs")
print(f"Exclusion rate:                 {len(excluded_pmids)/max(len(all_search_pmids),1)*100:.1f}%")

# Build excluded (review_doi, pmid) pairs for ground truth
excluded_pairs = []
for pmid in excluded_pmids:
    for rdoi in pmid_to_reviews[pmid]:
        excluded_pairs.append({'review_doi': rdoi, 'pmid': pmid})

excluded_pairs_df = pd.DataFrame(excluded_pairs).drop_duplicates()
print(f"\nExcluded (review, pmid) pairs:  {len(excluded_pairs_df):,}")
print(f"Unique reviews with excluded:   {excluded_pairs_df['review_doi'].nunique():,}")

# Store the relevance order for each review (PMIDs appear in relevance-sorted order)
# We'll use this later to take top-N most relevant per review
relevance_order = {}
for _, row in successful.iterrows():
    if pd.isna(row['pmids_json']) or row['pmids_json'] == '[]':
        continue
    pmids = json.loads(row['pmids_json'])
    review_dois = row['all_review_dois'].split('|')
    for rdoi in review_dois:
        if rdoi not in relevance_order:
            relevance_order[rdoi] = []
        # Extend in order (first = most relevant)
        relevance_order[rdoi].extend([p for p in pmids if p not in set(relevance_order[rdoi])])

print(f"\nRelevance ordering captured for {len(relevance_order)} reviews")

IDENTIFYING EXCLUDED PAPERS
Total queries executed: 40
  With results: 36
  Errors: 0
  Hit retmax cap (100,000): 11

After quality filter (non-empty, no errors):
  Retained queries: 36
  Of which hit retmax cap: 11
  Dropped (empty/errored): 4

Result count stats for retained queries:
count    3.600000e+01
mean     6.047460e+05
std      1.375169e+06
min      4.000000e+00
25%      2.660750e+03
50%      2.914750e+04
75%      2.240348e+05
max      5.252750e+06



Total unique PMIDs from filtered searches: 228,916

Overlap with Cochrane reviews:  1,134 PMIDs
Excluded (search-only):         227,782 PMIDs
Exclusion rate:                 99.5%

Excluded (review, pmid) pairs:  238,307
Unique reviews with excluded:   35

Relevance ordering captured for 35 reviews


In [8]:
# =============================================================================
# Fetch Abstracts for Excluded Papers
# =============================================================================

print("FETCHING ABSTRACTS FOR EXCLUDED PAPERS")
print("=" * 60)

pmids_to_fetch = list(excluded_pmids)
print(f"Fetching abstracts for all {len(pmids_to_fetch):,} excluded PMIDs")

est_batches = len(pmids_to_fetch) // 200 + 1
est_min = est_batches * NCBI_RATE / 60
print(f"Estimated: {est_batches:,} batches, ~{max(est_min, 1):.0f} min\n")

start_time = time.time()
excluded_records = fetch_abstracts_batch(pmids_to_fetch, batch_size=200)
elapsed = time.time() - start_time

with_abstract = sum(1 for r in excluded_records.values() if r.get('abstract'))
print(f"\nFetched {len(excluded_records):,} records in {elapsed/60:.1f} min")
print(f"  With abstracts:    {with_abstract:,} ({with_abstract/max(len(excluded_records),1)*100:.1f}%)")
print(f"  Without abstracts: {len(excluded_records) - with_abstract:,}")

FETCHING ABSTRACTS FOR EXCLUDED PAPERS
Fetching abstracts for all 227,782 excluded PMIDs
Estimated: 1,139 batches, ~2 min



Fetching abstracts:   0%|          | 0/1139 [00:00<?, ?it/s]


Fetched 227,499 records in 60.6 min
  With abstracts:    195,247 (85.8%)
  Without abstracts: 32,252


In [9]:
# =============================================================================
# Build & Save Final Output
# =============================================================================

print("BUILDING EXCLUDED PAPERS DATASET")
print("=" * 60)

# Join excluded pairs with fetched abstracts
output_rows = []
no_record = 0

for _, pair in tqdm(excluded_pairs_df.iterrows(), total=len(excluded_pairs_df),
                     desc="Building output"):
    pmid = pair['pmid']
    rec = excluded_records.get(pmid)
    if rec is None:
        no_record += 1
        continue

    output_rows.append({
        'review_doi': pair['review_doi'],
        'pmid': pmid,
        'title': rec.get('title', ''),
        'abstract': rec.get('abstract', ''),
        'authors': rec.get('authors', ''),
        'year': rec.get('year', ''),
        'doi': rec.get('doi', ''),
    })

excluded_df = pd.DataFrame(output_rows)

# Filter to rows that actually have an abstract (required for LLM evaluation)
excluded_with_abs = excluded_df[
    excluded_df['abstract'].notna() &
    (excluded_df['abstract'].str.len() > 50)
].copy()

print(f"\nTotal excluded pairs with records: {len(excluded_df):,}")
print(f"With abstracts (>50 chars):        {len(excluded_with_abs):,}")

# ── FILTER: Remove papers published AFTER the review ──────────────────────────────────
# Papers published after a review couldn't have been screened
excluded_with_abs['paper_year'] = pd.to_numeric(excluded_with_abs['year'], errors='coerce')
excluded_with_abs['review_year'] = excluded_with_abs['review_doi'].map(review_year_map)

before_filter = len(excluded_with_abs)
excluded_with_abs = excluded_with_abs[
    excluded_with_abs['paper_year'].isna() |  # keep if year unknown
    excluded_with_abs['review_year'].isna() |  # keep if review year unknown
    (excluded_with_abs['paper_year'] <= excluded_with_abs['review_year'])
].copy()
print(f"After date filter (paper_year <= review_year): {len(excluded_with_abs):,}")
print(f"  Removed (published after review): {before_filter - len(excluded_with_abs):,}")

# ── SAMPLE: Proportional excluded per review (80% excluded / 20% included) ───
# Instead of a fixed cap, scale excluded count so that each review's final
# dataset is 80% excluded and 20% included:  n_excluded = 4 × n_included
TARGET_EXCLUDED_FRAC = 0.80  # desired fraction of excluded in each review's dataset

# Count included papers per review (matching NB03 logic: abstract > 50 chars, deduped)
_incl_df = pd.read_csv(ABSTRACTS_CSV, usecols=['study_id', 'review_doi', 'abstract'], dtype=str)
_incl_df = _incl_df[_incl_df['abstract'].notna() & (_incl_df['abstract'].str.len() > 50)]
_incl_df = _incl_df.drop_duplicates(subset=['study_id', 'review_doi'], keep='first')
included_per_review = _incl_df.groupby('review_doi').size().to_dict()
del _incl_df

# Calculate per-review excluded cap:  n_excluded = n_included × ratio / (1 − ratio)
_multiplier = TARGET_EXCLUDED_FRAC / (1 - TARGET_EXCLUDED_FRAC)  # 0.80/0.20 = 4.0
excluded_cap = {rdoi: max(1, int(n * _multiplier))
                for rdoi, n in included_per_review.items()}

print(f"\nProportional sampling target: {TARGET_EXCLUDED_FRAC:.0%} excluded / {1-TARGET_EXCLUDED_FRAC:.0%} included")
print(f"  Multiplier: {_multiplier:.1f}× included count")
caps = pd.Series(excluded_cap)
print(f"  Per-review cap — Min: {caps.min()}, Max: {caps.max()}, Mean: {caps.mean():.0f}")

def get_relevance_rank(row):
    """Get relevance rank for a paper within its review (lower = more relevant)"""
    rdoi = row['review_doi']
    pmid = row['pmid']
    if rdoi in relevance_order:
        try:
            return relevance_order[rdoi].index(pmid)
        except ValueError:
            return 999999
    return 999999

excluded_with_abs['relevance_rank'] = excluded_with_abs.apply(get_relevance_rank, axis=1)

# Take proportional number of most relevant per review
before_sample = len(excluded_with_abs)
excluded_with_abs = excluded_with_abs.sort_values('relevance_rank')
excluded_with_abs['_cap'] = excluded_with_abs['review_doi'].map(excluded_cap).fillna(1).astype(int)
excluded_with_abs['_rank'] = excluded_with_abs.groupby('review_doi').cumcount()
excluded_with_abs = excluded_with_abs[excluded_with_abs['_rank'] < excluded_with_abs['_cap']].copy()
excluded_with_abs = excluded_with_abs.drop(columns=['_cap', '_rank']).reset_index(drop=True)

print(f"\nAfter proportional sampling per review: {len(excluded_with_abs):,}")
print(f"  Removed (lower relevance): {before_sample - len(excluded_with_abs):,}")

# Distribution per review
per_review = excluded_with_abs.groupby('review_doi').size()
print(f"\nExcluded papers per review:")
print(f"  Min: {per_review.min()}, Max: {per_review.max()}, Mean: {per_review.mean():.0f}")

# Clean up helper columns
excluded_with_abs = excluded_with_abs.drop(columns=['paper_year', 'review_year', 'relevance_rank'], errors='ignore')

print(f"Skipped (no PubMed record):        {no_record:,}")
print(f"\nUnique excluded PMIDs with abstract: {excluded_with_abs['pmid'].nunique():,}")
print(f"Unique reviews represented:          {excluded_with_abs['review_doi'].nunique():,}")

# Save
excluded_with_abs.to_csv(EXCLUDED_ABSTRACTS_CSV, index=False)
print(f"\n✓ Saved to {EXCLUDED_ABSTRACTS_CSV.name}")

# Also save the search results summary (without PMIDs JSON for smaller file)
search_summary = results_df.drop(columns=['pmids_json'], errors='ignore')
search_summary.to_csv(SEARCH_RESULTS_CSV, index=False)
print(f"✓ Search summary saved to {SEARCH_RESULTS_CSV.name}")

BUILDING EXCLUDED PAPERS DATASET


Building output:   0%|          | 0/238307 [00:00<?, ?it/s]


Total excluded pairs with records: 238,008
With abstracts (>50 chars):        204,716
After date filter (paper_year <= review_year): 94,967
  Removed (published after review): 109,749

Proportional sampling target: 80% excluded / 20% included
  Multiplier: 4.0× included count
  Per-review cap — Min: 8, Max: 1948, Mean: 320

After proportional sampling per review: 9,767
  Removed (lower relevance): 85,200

Excluded papers per review:
  Min: 1, Max: 1686, Mean: 287
Skipped (no PubMed record):        299

Unique excluded PMIDs with abstract: 9,648
Unique reviews represented:          34

✓ Saved to pubmed_excluded_abstracts.csv
✓ Search summary saved to pubmed_search_results.csv


In [10]:
# =============================================================================
# Summary Statistics
# =============================================================================

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)

print(f"\nInput:")
print(f"  Translated search queries:     {len(unique_queries):,}")
print(f"  Known PMIDs (Cochrane refs):   {len(known_pmids):,}")

print(f"\nPubMed Searches:")
print(f"  Queries executed:              {len(results_df):,}")
print(f"  Successful:                    {len(successful):,}")
print(f"  Unique PMIDs found:            {len(all_search_pmids):,}")

print(f"\nExcluded Papers:")
print(f"  Excluded PMIDs (total):        {len(excluded_pmids):,}")
print(f"  With abstracts fetched:        {excluded_with_abs['pmid'].nunique():,}")
print(f"  (review, paper) pairs:         {len(excluded_with_abs):,}")

print(f"\nOutput Files:")
print(f"  {EXCLUDED_ABSTRACTS_CSV.name}")
print(f"  {SEARCH_RESULTS_CSV.name}")

print(f"\n✓ Ready for notebook 03 (build ground truth)")

PIPELINE SUMMARY

Input:
  Translated search queries:     40
  Known PMIDs (Cochrane refs):   9,244

PubMed Searches:
  Queries executed:              40
  Successful:                    36
  Unique PMIDs found:            228,916

Excluded Papers:
  Excluded PMIDs (total):        227,782
  With abstracts fetched:        9,648
  (review, paper) pairs:         9,767

Output Files:
  pubmed_excluded_abstracts.csv
  pubmed_search_results.csv

✓ Ready for notebook 03 (build ground truth)
